# 07 — AdS/CFT foundations and the simplest Ryu-Takayanagi check

**Spacetime Lab — Phase 7 concept + implementation notebook**

This is where six phases of preparatory work pay off. For five phases we built classical general relativity (Schwarzschild, Penrose diagrams, Kerr, horizon finders, ringdown). Phase 6 built quantum information primitives (density matrices, von Neumann entropy, Schmidt decomposition). Phase 7 welds the two together via the **holographic entanglement entropy correspondence**.

**The gate test** of this notebook is the simplest non-trivial check of the Ryu-Takayanagi formula:

$$\frac{\text{Length}(\gamma_A^{\text{bulk geodesic}})}{4 G_N} \;\stackrel{?}{=}\; \frac{c}{3} \log\!\left(\frac{L_A}{\epsilon}\right) \;=\; S_A^{\text{boundary CFT}}.$$

The bulk side is computed from a closed-form Poincaré-AdS$_3$ geodesic length. The boundary side is the Calabrese-Cardy 2004 entanglement entropy of an interval in a 2D CFT, computed entirely from CFT physics with no reference to gravity. The two must agree to *machine precision* when the central charge is determined from the AdS radius via Brown-Henneaux. **They do — exactly.**

**What you will learn**

1. The pure anti-de Sitter spacetime in Poincaré coordinates and its conformal boundary
2. The constant-negative-curvature property: $R_{\mu\nu} = -(n-1)/L^2 \, g_{\mu\nu}$ verified numerically
3. The Brown-Henneaux relation $c = 3 L / (2 G_N)$ tying the bulk gravitational scale to the boundary CFT central charge
4. Geodesics in Poincaré AdS$_3$ are semicircles in the upper half-plane, with closed-form regularised length $2 L \log(L_A/\epsilon)$
5. The Ryu-Takayanagi formula and its agreement with the Calabrese-Cardy 2004 result
6. Numerical verification at five different sets of parameters — bit-exact agreement

**References**

- Brown & Henneaux, *Comm. Math. Phys.* **104** 207 (1986) — central charge
- Calabrese & Cardy, *J. Stat. Mech.* 0406:P06002 (2004), [arXiv:hep-th/0405152](https://arxiv.org/abs/hep-th/0405152)
- Ryu & Takayanagi, *Phys. Rev. Lett.* **96** 181602 (2006), [arXiv:hep-th/0603001](https://arxiv.org/abs/hep-th/0603001)
- Maldacena, *Adv. Theor. Math. Phys.* **2** 231 (1998)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.metrics import AdS
from spacetime_lab.holography import (
    geodesic_length_ads3,
    brown_henneaux_central_charge,
    ryu_takayanagi_ads3,
    calabrese_cardy_2d,
    verify_rt_against_calabrese_cardy,
)

plt.rcParams['figure.figsize'] = (8, 5)

## 1. The AdS metric

Anti-de Sitter spacetime AdS$_n$ is the maximally symmetric Lorentzian manifold with constant negative curvature. It is the unique solution of the vacuum Einstein equations with negative cosmological constant

$$\Lambda = -\frac{(n-1)(n-2)}{2 L^2}.$$

The two standard coordinate systems are global coordinates and Poincaré coordinates. We use the latter:

$$ds^2 = \frac{L^2}{z^2}\big(-dt^2 + dx_1^2 + \cdots + dx_{n-2}^2 + dz^2\big), \qquad z > 0.$$

The conformal boundary is at $z \to 0^+$ — a copy of $(n-1)+1$-dimensional Minkowski. **The dual CFT lives there.** For AdS$_3$ (the case we use), the boundary is 2D Minkowski and the dual is a 1+1-dimensional CFT.

Spatial slices of AdS$_3$ are 2D hyperbolic spaces of curvature $-1/L^2$ — precisely the upper half-plane model of $\mathbb{H}^2$ (rescaled by $L^2$).

In [ ]:
ads = AdS(dimension=3, radius=1.0)
print(f'{ads}')
print(f'Cosmological constant Lambda = {ads.cosmological_constant():.6f}  (expect -1/L^2 = -1)')
print(f'Expected Ricci scalar R = {ads.expected_ricci_scalar():.6f}  (expect -n(n-1)/L^2 = -6)')
print()
print('Line element:')
print(' ', ads.line_element_latex())

## 2. Numerical verification of the Einstein constant-curvature equation

The strong test of the line element: $R_{\mu\nu} - c\, g_{\mu\nu} = 0$ pointwise, where $c = -(n-1)/L^2$. If any component of the metric were miswritten, this residual would not vanish.

Following the Phase 3 lesson on Kerr (sympy `simplify` is pathologically slow on non-trivial metrics), the verification lambdifies the Ricci tensor and the metric tensor as numerical functions and evaluates the residual at sample points. Typical runtime: ~0.1 seconds for AdS$_3$, ~0.3 seconds for AdS$_4$.

In [ ]:
import time

for dim in [3, 4, 5]:
    for radius in [1.0, 2.0]:
        ads = AdS(dimension=dim, radius=radius)
        t0 = time.time()
        residual = ads.verify_einstein_constant_curvature()
        elapsed = time.time() - t0
        print(f'AdS_{dim} (L={radius}):  max |R_munu - c g_munu| = {residual:.3e}   ({elapsed:.2f}s)')

All zero residuals. The line element is correct in AdS$_3$, AdS$_4$, AdS$_5$ at multiple radii. We can now safely use AdS as a backdrop for the holographic computation.

## 3. The Brown-Henneaux relation

Derived in 1986 by Brown and Henneaux from the asymptotic symmetry algebra of asymptotically AdS$_3$ spacetimes:

$$c = \frac{3 L}{2 G_N}.$$

The boundary CFT central charge is determined entirely by the AdS radius and Newton's constant. This was discovered ~12 years before AdS/CFT was formalised by Maldacena in 1997, and is one of the deepest hints that AdS$_3$ is dual to a 2D CFT.

Setting $G_N = 1$ (geometric units, consistent with Phases 1-6):

| AdS radius $L$ | Central charge $c$ | Notable example |
|---|---|---|
| $2/3$ | $1$ | Free boson |
| $1$ | $3/2$ | (no canonical name) |
| $4/3$ | $2$ | Dirac fermion or two free bosons |
| $4$ | $6$ | $E_6$ minimal model? |
| $\rightarrow \infty$ | $\rightarrow \infty$ | Classical / large-$c$ limit |

In [ ]:
for L in [2/3, 1.0, 4/3, 2.0, 4.0]:
    c = brown_henneaux_central_charge(radius=L, G_N=1.0)
    print(f'L = {L:6.4f}   c = {c:.4f}')

## 4. Geodesics in Poincaré AdS$_3$

On a constant-time slice of Poincaré AdS$_3$, the induced metric is

$$ds^2_{\text{slice}} = L^2 \cdot \frac{dx^2 + dz^2}{z^2}.$$

This is the **upper half-plane model of 2D hyperbolic space**, $\mathbb{H}^2$, with constant negative curvature $-1/L^2$.

**Geodesics in $\mathbb{H}^2$** are well-known: vertical half-lines and **semicircles whose centre lies on the boundary $z = 0$**. For two boundary anchor points $(x_A, 0)$ and $(x_B, 0)$, the geodesic is the semicircle of centre $(x_A + x_B)/2$ and radius $|x_B - x_A|/2$.

Let's draw a few of these:

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Several boundary intervals of different lengths
for x_A, x_B, color in [
    (-3, 3, '#0050a0'),
    (-2, 1, '#1a9a4a'),
    (0.5, 2.5, '#c64a0b'),
    (-4.5, -3.0, '#7030a0'),
]:
    centre = (x_A + x_B) / 2
    R = abs(x_B - x_A) / 2
    theta = np.linspace(0, np.pi, 200)
    x = centre + R * np.cos(theta)
    z = R * np.sin(theta)
    ax.plot(x, z, color=color, lw=2.0,
            label=f'$(x_A, x_B) = ({x_A}, {x_B})$, $L_A = {2*R}$')
    # Mark the boundary anchors
    ax.plot([x_A, x_B], [0, 0], 'o', color=color, markersize=6)

# Draw the boundary z = 0
ax.axhline(0, color='k', lw=1.5)
ax.text(5, -0.15, 'conformal boundary $z = 0$', ha='right', fontsize=10)

ax.set_xlim(-5.5, 5.5)
ax.set_ylim(-0.3, 3.5)
ax.set_xlabel(r'$x$ (boundary direction)')
ax.set_ylabel(r'$z$ (radial direction into the bulk)')
ax.set_title(r'Geodesics in Poincar\'e AdS$_3$ (upper half-plane $\mathbb{H}^2$)')
ax.set_aspect('equal')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Each semicircle is a bulk geodesic anchored to the two boundary points marked with dots. The longer the boundary interval, the larger the semicircle and the deeper it dips into the bulk.

## 5. Regularised geodesic length

The actual geodesic is infinite — it asymptotes to the boundary at each end. We need a UV cutoff $\epsilon$ that says "stop when $z = \epsilon$". The closed-form result is

$$\mathcal{L}(\epsilon) = 2 L \log\!\left(\frac{|x_B - x_A|}{\epsilon}\right).$$

The factor of $L$ at the front is because the metric is $L^2$ times the standard $\mathbb{H}^2$ metric. The factor of 2 comes from going around the semicircle (each half contributes $L \log(\dots)$). This formula is the **bulk-side input** to Ryu-Takayanagi.

In [ ]:
# Tabulate for several intervals
L_ads = 1.0
epsilon = 0.01
print(f'{"L_A":>6s}  {"closed form":>14s}  {"= 2 L log(L_A/eps)":>22s}')
print('-' * 50)
for L_A in [1.0, 2.0, 5.0, 10.0]:
    geo_length = geodesic_length_ads3(0.0, L_A, radius=L_ads, epsilon=epsilon)
    explicit = 2 * L_ads * math.log(L_A / epsilon)
    print(f'{L_A:6.2f}  {geo_length:14.6f}  {explicit:22.6f}')

## 6. THE Ryu-Takayanagi formula

$$S_A = \frac{\text{Length}(\gamma_A^{\text{bulk}})}{4 G_N}.$$

Conjectured by Ryu and Takayanagi in 2006 (Phys. Rev. Lett. 96, 181602), this formula says that the entanglement entropy of a spatial subregion in the boundary CFT equals one quarter of the area (in 4d AdS, length in 3d AdS) of a *minimal* bulk surface anchored to the boundary of $A$.

**For Poincaré AdS$_3$/CFT$_2$ with a single interval**, the minimal surface is the unique semicircular geodesic, so

$$S_A^{\text{RT}} = \frac{2 L \log(L_A/\epsilon)}{4 G_N} = \frac{L}{2 G_N}\log(L_A/\epsilon).$$

Substituting Brown-Henneaux $c = 3 L / (2 G_N)$, this becomes

$$S_A^{\text{RT}} = \frac{c}{3} \log(L_A/\epsilon),$$

which is **exactly the Calabrese-Cardy 2004 formula for the entanglement entropy of an interval in a 2D CFT**, derived independently from a CFT replica-trick computation with no reference to gravity.

## 7. The gate test — bulk RT vs boundary CC

The two must agree numerically. Let's verify.

In [ ]:
print(f'{"L_A":>6s}  {"L_AdS":>6s}  {"eps":>10s}  {"G_N":>6s}  {"S_RT":>14s}  {"S_CC":>14s}  {"residual":>12s}')
print('-' * 80)
for L_A, L_ads, eps, G in [
    (1.0, 1.0, 0.01, 1.0),
    (2.0, 1.0, 0.001, 1.0),
    (5.0, 2.0, 0.001, 1.0),
    (10.0, 4.0, 1e-5, 1.0),
    (3.7, 2.5, 0.0001, 0.5),
]:
    rt, cc, res = verify_rt_against_calabrese_cardy(L_A, L_ads, eps, G)
    print(f'{L_A:6.2f}  {L_ads:6.2f}  {eps:10.2e}  {G:6.2f}  {rt:14.10f}  {cc:14.10f}  {res:12.2e}')

**Residual = exactly 0.00e+00 in every case.** Not "machine precision" — *bit-exact*. This is because the two code paths reduce to the same floating-point computation `L/(2G) * log(L_A/eps)`, just with the substitutions applied in different orders.

**This is the simplest non-trivial verification of the holographic entanglement entropy correspondence.** A bulk geometric computation (geodesic length in AdS$_3$) and a boundary quantum-information computation (entanglement entropy of a 2D CFT interval) give *the same number*. The factor of $1/(4 G_N)$ that converts geometric area to entropy — the same factor that appears in $S_{BH} = A/(4 G_N)$ for a Schwarzschild black hole — is universal.

## 8. The interval-length scaling — universal logarithmic divergence

Both formulas predict $S_A \propto \log(L_A/\epsilon)$, with slope $c/3$. Let's plot it against $L_A$ to see the scaling and verify it numerically over a wide range.

In [ ]:
L_ads = 1.0  # AdS radius
G_N = 1.0
epsilon = 1e-4
c = brown_henneaux_central_charge(radius=L_ads, G_N=G_N)

L_A_values = np.logspace(-2, 3, 100)
S_rt = [ryu_takayanagi_ads3(L_A, radius=L_ads, epsilon=epsilon, G_N=G_N)
        for L_A in L_A_values]
S_cc = [calabrese_cardy_2d(L_A, central_charge=c, epsilon=epsilon)
        for L_A in L_A_values]

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(L_A_values, S_rt, color='#0050a0', lw=2.5, label='bulk RT (AdS$_3$ geodesic / $4G_N$)')
ax.plot(L_A_values, S_cc, color='#c64a0b', lw=1.0, ls='--', label='boundary CFT (Calabrese-Cardy)')
ax.set_xscale('log')
ax.set_xlabel(r'interval length $L_A$ (log scale)')
ax.set_ylabel(r'entanglement entropy $S_A$ (nats)')
ax.set_title(f'Holographic entanglement entropy: c = {c:.3f}, $\\epsilon$ = {epsilon}')
ax.legend(loc='upper left')
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.show()

max_diff = max(abs(rt - cc) for rt, cc in zip(S_rt, S_cc))
print(f'\nMax difference between bulk and boundary computations: {max_diff:.2e}')

The two curves are **literally indistinguishable** — they overlap exactly across 5 orders of magnitude in $L_A$. The maximum difference is at floating-point bit precision (zero or a few ULPs).

This is what holographic entanglement entropy looks like in the simplest possible case. **The entire content of the holographic dictionary for this case is the factor $1/(4 G_N)$ converting bulk length to boundary entropy, times the Brown-Henneaux relation $c = 3 L / (2 G_N)$ converting bulk parameters to boundary central charge.** That's it.

## 9. Phase 7 gate checks

Before we move on to Phase 8 the following must hold.

In [ ]:
# (1) AdS_3 unit radius is Einstein-constant with R_munu = -2 g_munu
ads = AdS(dimension=3, radius=1.0)
assert ads.verify_einstein_constant_curvature() < 1e-10
assert math.isclose(ads.expected_ricci_scalar(), -6.0)
assert math.isclose(ads.cosmological_constant(), -1.0)

# (2) AdS_4 also Einstein-constant
ads4 = AdS(dimension=4, radius=1.5)
assert ads4.verify_einstein_constant_curvature() < 1e-10

# (3) Brown-Henneaux: free-boson c=1 from L = 2/3
assert math.isclose(brown_henneaux_central_charge(radius=2.0/3.0), 1.0)

# (4) Geodesic length closed form
L_geo = geodesic_length_ads3(0.0, 2.0, radius=1.0, epsilon=0.01)
assert math.isclose(L_geo, 2 * math.log(200.0))

# (5) Ryu-Takayanagi for an interval
S_rt = ryu_takayanagi_ads3(2.0, radius=1.0, epsilon=0.01)
assert math.isclose(S_rt, 0.5 * math.log(200.0))

# (6) Calabrese-Cardy boundary formula
S_cc = calabrese_cardy_2d(2.0, central_charge=1.5, epsilon=0.01)
assert math.isclose(S_cc, 0.5 * math.log(200.0))

# (7) RT == CC via Brown-Henneaux for a wide range of parameters
for L_A, L_ads_, eps, G in [
    (1.0, 1.0, 0.01, 1.0),
    (2.0, 1.0, 0.001, 1.0),
    (5.0, 2.0, 0.001, 1.0),
    (10.0, 4.0, 1e-5, 1.0),
    (100.0, 1.0, 1e-6, 1.0),
]:
    _, _, residual = verify_rt_against_calabrese_cardy(L_A, L_ads_, eps, G)
    assert residual < 1e-12

# (8) The two computations are bit-identical for unit inputs
_, _, res = verify_rt_against_calabrese_cardy(1.0, 1.0, 0.01)
assert res == 0.0

print('ALL PHASE 7 GATE CHECKS PASSED')

---

## What's next — Phase 8 teaser

Phase 7 handled the simplest case: a single interval in pure Poincaré AdS$_3$. Phase 8 generalises in three directions:

1. **Two-interval entanglement** — the entropy of $A_1 \cup A_2$ undergoes a phase transition between connected and disconnected geodesic configurations as the intervals are pulled apart. This is the simplest holographic phase transition and the canonical demonstration of mutual-information dynamics.
2. **BTZ black hole** — the AdS$_3$ analogue of Schwarzschild. Has horizons, finite temperature, and a Hawking-Page transition. The RT formula at finite temperature gives the *thermal* entropy of the boundary CFT, including the Bekenstein-Hawking-Cardy formula $S = (c/3) \log(\sinh(\pi L_A / \beta)/\sinh(\pi \epsilon / \beta))$ at temperature $T = 1/\beta$.
3. **Numerical minimal-surface finder** — for higher-dimensional AdS bulks where there is no closed-form geodesic, we need an actual minimal-surface finder. This is the most code-heavy part of Phase 8 but conceptually straightforward (a relaxation method on a surface).

Phase 9 then handles the *island formula* for evaporating black holes — the modern frontier of holographic entanglement.

## 10. Honest scope note

This notebook implements the **simplest non-trivial test** of holographic entanglement entropy. We deliberately:

- Used the closed-form Poincaré AdS$_3$ geodesic length, not a numerical bulk minimal-surface finder (deferred to Phase 8)
- Restricted to a single interval, not the two-interval phase transition (deferred to Phase 8)
- Did not implement the BTZ black hole or finite-temperature ringdown corrections (deferred to Phase 8)
- Did not touch the island formula or quantum corrections (deferred to Phase 9)

These are real, hard problems and they justify their own phases. Phase 7 establishes the foundation: **the bulk and boundary computations agree, exactly, in the simplest case**, which is the existence proof that the rest of the program is well-posed.